# 📓 Notebook 2: EDA + Modelos de Partido (2A Regresión | 2B Clasificación)
**Machine Learning I — Taller 2 | Universidad Externado de Colombia**  
**Integrantes:** Miguel Ángel Camargo | Camilo Hernández  
**Docente:** Julián Zuluaga

---
## Objetivo
Explorar los datos de partidos y entrenar:
- **Modelo 2A**: Regresión lineal para predecir `total_goals`
- **Modelo 2B**: Clasificación multinomial para predecir el resultado `FTR` (H/D/A) superando a Bet365


## 1. Configuración e importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                              mean_absolute_error, mean_squared_error, r2_score, f1_score)
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

PROCESSED = Path('../data/processed')
OUTPUTS   = Path('../data/outputs')
print("✅ Librerías cargadas correctamente")


## 2. Carga de datos

In [ ]:
df = pd.read_csv(PROCESSED / 'features_modelo2.csv', parse_dates=['match_date'])
df['match_date'] = pd.to_datetime(df['match_date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['match_date']).sort_values('match_date').reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Rango de fechas: {df['match_date'].min().date()} → {df['match_date'].max().date()}")
df.head()


## 3. EDA — Análisis Exploratorio

In [ ]:
# Distribución de FTR
ftr_counts = df['ftr'].value_counts()
print("Distribución de resultados (FTR):")
print(ftr_counts)
print(ftr_counts / len(df) * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ftr_counts.plot(kind='bar', ax=axes[0], color=['#4A8C3F','#888','#C5A059'], edgecolor='white')
axes[0].set_title('Distribución de resultados FTR')
axes[0].set_xticklabels(['Local (H)', 'Visitante (A)', 'Empate (D)'], rotation=0)

df['total_goals'].hist(ax=axes[1], bins=range(0, 10), color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de goles totales por partido')
axes[1].set_xlabel('Total goles')
plt.tight_layout()
plt.show()


In [ ]:
# Cuotas Bet365 vs resultado real
if 'b365h' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    for col, label, color in [('b365h','Local (H)','#4A8C3F'),
                               ('b365d','Empate (D)','#888888'),
                               ('b365a','Visitante (A)','#C5A059')]:
        if col in df.columns:
            df[col].plot(kind='kde', ax=ax, label=label, color=color)
    ax.set_xlabel('Cuota Bet365')
    ax.set_title('Distribución de cuotas Bet365 por resultado')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Columnas b365h/d/a no encontradas en el dataset")


In [ ]:
# Forma reciente: xg_form_diff por resultado
if 'xg_form_diff' in df.columns:
    fig, ax = plt.subplots(figsize=(9, 4))
    for res, color in [('H','#4A8C3F'), ('D','#888888'), ('A','#C5A059')]:
        df[df['ftr'] == res]['xg_form_diff'].plot(kind='kde', ax=ax, label=res, color=color)
    ax.set_xlabel('xg_form_diff (home - away xG rolling 5)')
    ax.set_title('Distribución de xg_form_diff por resultado')
    ax.axvline(0, color='black', lw=0.8, linestyle='--')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 4. Modelo 2A — Regresión lineal: total_goals

In [ ]:
WINDOW = 5
REGRESSION_FEATURES = [
    f'home_xg_for_avg{WINDOW}', f'away_xg_for_avg{WINDOW}',
    f'home_sot_avg{WINDOW}', f'away_sot_avg{WINDOW}',
    'expected_total_xg', 'bookmaker_spread_draw', 'implied_prob_d', 'away_strength_rating',
]
reg_feats = [f for f in REGRESSION_FEATURES if f in df.columns]
TARGET_REG = 'total_goals'
print(f"Features usadas ({len(reg_feats)}):", reg_feats)


In [ ]:
# VIF Modelo 2A
df_reg = df[reg_feats + [TARGET_REG]].dropna()
X_reg = df_reg[reg_feats].astype(float)
vif_reg = pd.DataFrame({'feature': X_reg.columns,
                         'VIF': [variance_inflation_factor(X_reg.values, i)
                                 for i in range(X_reg.shape[1])]})
print(vif_reg.sort_values('VIF', ascending=False).to_string(index=False))


In [ ]:
# CV temporal — Modelo 2A
df_reg_sorted = df[reg_feats + [TARGET_REG, 'match_date']].dropna().sort_values('match_date')
X = df_reg_sorted[reg_feats].values
y = df_reg_sorted[TARGET_REG].values
tscv = TimeSeriesSplit(n_splits=5)

r2s, rmses, maes = [], [], []
for train_idx, test_idx in tscv.split(X):
    pipe_r = Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())])
    pipe_r.fit(X[train_idx], y[train_idx])
    pred = pipe_r.predict(X[test_idx])
    r2s.append(r2_score(y[test_idx], pred))
    rmses.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
    maes.append(mean_absolute_error(y[test_idx], pred))

print(f"R²   (CV mean): {np.mean(r2s):.4f}")
print(f"RMSE (CV mean): {np.mean(rmses):.4f}")
print(f"MAE  (CV mean): {np.mean(maes):.4f}")
r2_status = "✅ > 0" if np.mean(r2s) > 0 else "❌ < 0 — modelo no aporta vs. media"
print(r2_status)


In [ ]:
# Análisis de residuos — Modelo 2A
cut = int(len(df_reg_sorted) * 0.8)
pipe_r_final = Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())])
pipe_r_final.fit(df_reg_sorted.iloc[:cut][reg_feats], df_reg_sorted.iloc[:cut][TARGET_REG])
pred_test = pipe_r_final.predict(df_reg_sorted.iloc[cut:][reg_feats])
residuals = df_reg_sorted.iloc[cut:][TARGET_REG].values - pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(pred_test, residuals, alpha=0.4, s=20, color='steelblue')
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Goles predichos')
axes[0].set_ylabel('Residuo')
axes[0].set_title('Residuos — Modelo 2A')

axes[1].hist(residuals, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', lw=1)
axes[1].set_xlabel('Residuo')
axes[1].set_title('Distribución de residuos')
plt.tight_layout()
plt.show()


## 5. Modelo 2B — Clasificación multinomial: FTR

In [ ]:
CLASSIFICATION_FEATURES = [
    'implied_prob_h', 'implied_prob_d', 'implied_prob_a',
    'xg_form_diff', 'points_form_diff', 'big_chances_diff', 'home_strength_rating',
]
clf_feats = [f for f in CLASSIFICATION_FEATURES if f in df.columns]
TARGET_CLF = 'ftr'
print(f"Features usadas ({len(clf_feats)}):", clf_feats)


In [ ]:
# CV temporal — Modelo 2B vs Bet365
df_clf = df[clf_feats + [TARGET_CLF, 'implied_prob_h', 'implied_prob_d', 'implied_prob_a', 'match_date']].dropna().sort_values('match_date')
X_clf = df_clf[clf_feats].values
y_clf = df_clf[TARGET_CLF].values
tscv = TimeSeriesSplit(n_splits=5)

accs, f1s, bet_accs = [], [], []
for train_idx, test_idx in tscv.split(X_clf):
    pipe_c = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
    ])
    pipe_c.fit(X_clf[train_idx], y_clf[train_idx])
    pred = pipe_c.predict(X_clf[test_idx])
    accs.append(accuracy_score(y_clf[test_idx], pred))
    f1s.append(f1_score(y_clf[test_idx], pred, average='macro'))

    subset = df_clf.iloc[test_idx]
    bet_pred = subset[['implied_prob_h','implied_prob_d','implied_prob_a']].idxmax(axis=1)
    bet_pred = bet_pred.map({'implied_prob_h':'H','implied_prob_d':'D','implied_prob_a':'A'})
    bet_accs.append(accuracy_score(y_clf[test_idx], bet_pred))

m_acc  = np.mean(accs)
b_acc  = np.mean(bet_accs)
print(f"Accuracy Modelo 2B : {m_acc*100:.2f}%")
print(f"Accuracy Bet365    : {b_acc*100:.2f}%")
print(f"Diferencia         : {(m_acc - b_acc)*100:+.2f} pp")
print(f"{'✅ Supera a Bet365' if m_acc > b_acc else '❌ Por debajo de Bet365'}")
print(f"F1-macro           : {np.mean(f1s):.4f}")


In [ ]:
# Comparativo visual Accuracy
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Modelo 2B\n(Logístico)', 'Bet365\n(Benchmark)'],
              [m_acc * 100, b_acc * 100], color=['#4A8C3F', '#C5A059'], edgecolor='white')
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=12)
ax.set_ylim(0, 70)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Modelo 2B vs Bet365 — Accuracy CV')
plt.tight_layout()
plt.show()


In [ ]:
# Matriz de confusión
cut_clf = int(len(df_clf) * 0.8)
pipe_c_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
])
pipe_c_final.fit(df_clf.iloc[:cut_clf][clf_feats], df_clf.iloc[:cut_clf][TARGET_CLF])
pred_clf = pipe_c_final.predict(df_clf.iloc[cut_clf:][clf_feats])
labels = ['H', 'D', 'A']
cm = confusion_matrix(df_clf.iloc[cut_clf:][TARGET_CLF], pred_clf, labels=labels)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, colorbar=False)
ax.set_title('Matriz de confusión — Modelo 2B')
plt.tight_layout()
plt.show()


In [ ]:
# Coeficientes Modelo 2B
coef_df = pd.DataFrame(
    pipe_c_final.named_steps['clf'].coef_,
    index=['A', 'D', 'H'],
    columns=clf_feats
).T
print("Coeficientes logísticos por clase:")
print(coef_df.to_string())


## 6. Conclusiones

### Modelo 2A — Regresión total_goals
| Métrica | Valor |
|---|---|
| R² (CV) | ~0 a negativo |
| RMSE | ~1.60 |
| MAE | ~1.27 |

> El R² bajo se explica por la ausencia de cuotas Over/Under de Bet365 en el dataset. Los goles son inherentemente difíciles de predecir sin esa señal.

### Modelo 2B — Clasificación FTR
| Métrica | Valor |
|---|---|
| Accuracy Modelo | **50.87%** |
| Accuracy Bet365 | **50.43%** |
| Diferencia | **+0.44 pp ✅** |
| F1-macro | ~0.35 |

> **Logramos superar a Bet365** combinando probabilidades implícitas + xg_form_diff (diferencial de xG rolling 5 partidos). La variable `xg_form_diff` es la contribución original más relevante.
